# Fox vision → dataset: can a fox learn to follow a sheep?

**Goal.** Prove that a small CNN+FNN can learn, from **only** what the fox perceives — its
egocentric **terrain** channel and its **entity** channel (the exposed sheep) — which
direction is the *correct* one to move to reach the sheep. This notebook builds the dataset.

We reuse the real simulation code unchanged (`sim/`): build a world, drop in **one fox** and
**one sheep** inside the fox's vision range, and run the real **perception → `RuleBrain`**
pipeline to get the *correct* heading (straight at the sheep).

**Directions, labelled.** For each scene we record the same two-channel view paired with
several candidate headings:

* the **true** heading — toward the sheep (`label = 1`), and
* a spread of **false** headings — *not* toward the sheep (`label = 0`): **sideways**
  (±90°), **oblique** (±135°), the **opposite** (180°), and a random off-angle.

So "wrong" is not just "the exact opposite" — sideways and oblique headings are wrong too,
and the dataset now says so. Nothing in `sim/` is modified — it is only imported.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the ecosystem package importable no matter which directory the kernel started in.
_here = Path.cwd()
REPO = next((c for c in [_here, *_here.parents]
             if (c / "config.py").exists() and (c / "sim").is_dir()), _here)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from config import make_config, SHEEP, FOX
from sim.world import World
from sim.environment import Environment
from sim.entities import Entities
from sim.grid import SpatialGrid
from sim.perception import Perception, FX_TERRAIN, FX_FOOD
from sim.brain import RuleBrain, A_DX, A_DY
from sim.systems import movement, vegetation
from sim import genome as gn

print("repo:", REPO)

## 1. Parameters and world

The perception window is fixed by the sim at `K = 2*R+1` (`R = 28`), but a fox only sees
within its own `sensory_range`. We pin the fox's eye radius to **12 cells** and crop the
stored window to ±12 (a `25×25` view), which is loss-less: everything outside the eye disc
is zero anyway.

The angular bands decide the labels: a candidate within **`POS_TOL`** of the true heading is
*true*; anything beyond **`NEG_MARGIN`** is *false* (the gap in between is left out so the
two classes are cleanly separated).

In [ ]:
# --- experiment parameters -------------------------------------------------
WORLD_SEED  = 12345      # fixes terrain + rivers (identical map every run)
FOX_SENSORY = 12.0       # fox eye radius in cells (gene range is [10, 28]); pinned here
CROP_R      = 12         # crop the egocentric window to +/- this many cells
WIN         = 2 * CROP_R + 1                      # -> 25 x 25
N_SCENARIOS = 1000       # each -> 1 true + several diverse false rows
GEN_SEED    = 20240709   # reproducible scenario sampling
DATA_PATH   = REPO / "notebooks" / "fox_vision_dataset.csv"

POS_TOL    = np.radians(20.0)   # within this of the true heading -> label 1
NEG_MARGIN = np.radians(45.0)   # beyond this from the true heading -> label 0
N_POS_JITTER = 1                # extra near-true positives per scene (smooths the target)

# --- build the headless world + the pieces perception needs ----------------
cfg   = make_config(world_seed=WORLD_SEED, seed=7)
world = World(cfg.world)
env   = Environment(cfg.env, np.random.default_rng(1))
ent   = Entities(cfg)
veg   = vegetation.initial_field(world, np.random.default_rng(2))   # sheep-food field
temp_field = env.temperature_field(world.static_temp)               # env frozen at t=0

# one spatial hash per species (perception queries them); rebuilt every scenario
grids = {SHEEP: SpatialGrid(world.w, world.h, cfg.sim.grid_cell_size),
         FOX:   SpatialGrid(world.w, world.h, cfg.sim.grid_cell_size)}
perc  = Perception(cfg, world, ent, env)
brain = RuleBrain(np.random.default_rng(3), cfg.sim.food_eat_threshold)

# candidate fox/sheep cells: passable, dry, and NOT forest cover. Sheep hidden in cover are
# invisible/uncatchable to foxes (the prey refuge), so they would carry no learning signal.
land_mask = world.passable & ~world.water_any & ~world.cover
land_ys, land_xs = np.nonzero(land_mask)

sheep_spec, fox_spec = cfg.species[SHEEP], cfg.species[FOX]
S_IDX = gn.GENE_INDEX["sensory_range"]
print(f"perception window K={perc.K}; stored crop {WIN}x{WIN}; land cells={land_xs.size}")

## 2. One scenario = one perception → true heading

`make_scenario` clears the world, places a fox and a sheep, and runs the real perception +
brain to get the **true** heading (toward the sheep). Two details from the sim:

* **The fox is made hungry.** The `RuleBrain` only *pursues* prey when a need is urgent; a
  well-fed fox just wanders. We set hunger high / thirst low so it commits to the sheep.
* **Grids are rebuilt and wired exactly as `Simulation.step` does**, so the fox's `entity`
  channel really contains the sheep.

In [ ]:
def rebuild_grids():
    """Re-index both species into their spatial hashes (mirrors Simulation._rebuild_grids)."""
    for sid, g in grids.items():
        sidx = np.nonzero(ent.species_mask(sid))[0]
        g.rebuild(sidx, ent.pos_x, ent.pos_y)


def make_scenario(rng):
    """Place ONE fox + ONE sheep in vision range; run perception -> RuleBrain.

    Returns (terrain, entity, dx, dy, meta), or None if no valid placement was found.
      terrain, entity : (WIN, WIN) cropped fox channels
      dx, dy          : unit TRUE heading the RuleBrain chose (points at the sheep)
    """
    ent.kill(ent.alive_indices())                       # fresh, empty world

    for _ in range(200):                                # sample a valid fox+sheep layout
        k = int(rng.integers(0, land_xs.size))
        fx, fy = land_xs[k] + 0.5, land_ys[k] + 0.5
        dist   = rng.uniform(2.0, FOX_SENSORY - 1.0)    # sheep strictly inside the eye disc
        ang    = rng.uniform(0.0, 2 * np.pi)
        sx, sy = fx + dist * np.cos(ang), fy + dist * np.sin(ang)
        scx, scy = int(sx), int(sy)
        if not (0 <= scx < world.w and 0 <= scy < world.h):
            continue
        if world.cover[scy, scx] or world.water_any[scy, scx]:
            continue
        break
    else:
        return None

    sg = gn.random_genomes(sheep_spec, 1, rng)
    fg = gn.random_genomes(fox_spec, 1, rng)
    fg[0, S_IDX] = FOX_SENSORY
    ent.spawn(sheep_spec, sg, np.array([[sx, sy]], np.float32), rng,
              energy=0.8, age=np.float32(sheep_spec.maturity_age * 2))
    fox_slot = int(ent.spawn(fox_spec, fg, np.array([[fx, fy]], np.float32), rng,
                             energy=0.5, age=np.float32(fox_spec.maturity_age * 2))[0])

    ent.hunger[fox_slot] = 0.75    # hungry fox -> RuleBrain locks onto the prey
    ent.thirst[fox_slot] = 0.05
    ent.energy[fox_slot] = 0.50

    rebuild_grids()
    perc._species_grids = grids
    perc.veg = veg
    perc.temp_field = temp_field
    obs_by_species, idx = perc.build()
    act = brain.decide(obs_by_species, idx)

    fgrids = obs_by_species[FOX].grids[0]               # (4, K, K); fox is the only row
    R = perc.R
    sl = slice(R - CROP_R, R + CROP_R + 1)
    terrain = fgrids[FX_TERRAIN, sl, sl].astype(np.float32).copy()
    entity  = fgrids[FX_FOOD,    sl, sl].astype(np.float32).copy()

    row = int(np.searchsorted(idx, fox_slot))
    dx, dy = float(act[row, A_DX]), float(act[row, A_DY])
    meta = dict(fox_slot=fox_slot, fx=fx, fy=fy, sx=sx, sy=sy,
                sheep_dx=sx - fx, sheep_dy=sy - fy, dist=float(dist),
                entity_cells=int((entity > 0).sum()))
    return terrain, entity, dx, dy, meta

## 3. Sanity check + visualization

Draw one scenario. The red arrow (the `RuleBrain`'s chosen heading) should point straight
at the sheep blip in the entity channel.

In [ ]:
terrain, entity, dx, dy, meta = make_scenario(np.random.default_rng(0))
sdx, sdy = meta["sheep_dx"], meta["sheep_dy"]
cos = (dx*sdx + dy*sdy) / (np.hypot(dx, dy)*np.hypot(sdx, sdy) + 1e-9)
print(f"true heading (dx,dy) = ({dx:+.3f}, {dy:+.3f})")
print(f"sheep offset (cells) = ({sdx:+.2f}, {sdy:+.2f})")
print(f"cosine(heading, true direction to sheep) = {cos:.3f}   (1.0 = perfect)")

c = CROP_R
fig, ax = plt.subplots(1, 2, figsize=(9, 4.4))
for a, grid, title in ((ax[0], terrain, "terrain channel"),
                       (ax[1], entity, "entity channel (sheep)")):
    a.imshow(grid, origin="upper", cmap="viridis")
    a.plot(c, c, "wo", ms=9, mec="k", label="fox")
    a.arrow(c, c, dx*4, dy*4, color="red", width=0.25, head_width=1.2,
            length_includes_head=True)
    a.set_title(title); a.legend(loc="upper right", fontsize=8)
fig.suptitle("Fox egocentric perception + TRUE heading (red arrow → sheep)")
plt.tight_layout(); plt.show()

## 4. Generate the dataset (true + diverse false headings)

For each scene we emit one row per candidate heading, all sharing the **same** perception:

| heading | offset from true | label |
|:--|:--|:--:|
| toward the sheep (+ a small jitter) | `≤ POS_TOL` | **1** |
| sideways | ≈ ±90° | 0 |
| oblique | ≈ ±135° | 0 |
| opposite | 180° | 0 |
| random off-angle | `≥ NEG_MARGIN` | 0 |

We also store `cos` = cosine between the row's heading and the true direction (≈ +1 for
true, ≈ 0 for sideways, ≈ −1 for opposite) — a graded "how good is this heading" signal.

In [ ]:
def candidate_headings(true_ang, rng):
    """(angle, label) pairs: the true heading (+jitter) and a spread of wrong ones."""
    cands = [(true_ang, 1)]
    for _ in range(N_POS_JITTER):
        cands.append((true_ang + rng.uniform(-POS_TOL, POS_TOL), 1))
    s = 1.0 if rng.random() < 0.5 else -1.0                    # random left/right per scene
    offsets = [s*np.radians(90), -s*np.radians(135), np.radians(180),   # sideways/oblique/opp
               s*rng.uniform(NEG_MARGIN, np.radians(180))]              # random off-angle
    cands += [(true_ang + off, 0) for off in offsets]
    return cands


rng = np.random.default_rng(GEN_SEED)
t_rows, e_rows, meta_rows = [], [], []
made = attempts = 0
while made < N_SCENARIOS:
    attempts += 1
    out = make_scenario(rng)
    if out is None:
        continue
    terrain, entity, dx, dy, meta = out
    if meta["entity_cells"] == 0 or (dx == 0.0 and dy == 0.0):
        continue                                        # sheep not perceived -> skip
    true_ang = np.arctan2(dy, dx)
    twx, twy = np.cos(true_ang), np.sin(true_ang)
    tf, ef = terrain.ravel(), entity.ravel()
    for ang, label in candidate_headings(true_ang, rng):
        cdx, cdy = float(np.cos(ang)), float(np.sin(ang))
        t_rows.append(tf); e_rows.append(ef)
        meta_rows.append((made, label, cdx, cdy, meta["sheep_dx"], meta["sheep_dy"],
                          meta["dist"], cdx*twx + cdy*twy))     # last col = cos vs true
    made += 1

t_arr = np.asarray(t_rows, np.float32)
e_arr = np.asarray(e_rows, np.float32)
meta_arr = np.asarray(meta_rows, np.float32)
print(f"{made} scenarios ({attempts} attempts) -> {len(t_rows)} rows; window {WIN}x{WIN}")
print("label balance:", dict(zip(*np.unique(meta_arr[:, 1].astype(int), return_counts=True))))

cols_meta = ["scenario_id", "label", "dx", "dy", "sheep_dx", "sheep_dy", "dist", "cos"]
cols_t = [f"t_{i}" for i in range(WIN*WIN)]
cols_e = [f"e_{i}" for i in range(WIN*WIN)]
df = pd.concat([
    pd.DataFrame(meta_arr, columns=cols_meta),
    pd.DataFrame(t_arr, columns=cols_t),
    pd.DataFrame(e_arr, columns=cols_e),
], axis=1)
df["scenario_id"] = df["scenario_id"].astype(int)
df["label"] = df["label"].astype(int)

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False, float_format="%.5f")
print("saved:", DATA_PATH, f"({DATA_PATH.stat().st_size/1e6:.1f} MB)")
df[["scenario_id", "label", "dx", "dy", "cos", "dist"]].head(8)

## 5. What's in the CSV

Columns:

* `scenario_id` — groups all candidate headings that share one perception (split on this).
* `label` — `1` = toward the sheep, `0` = a wrong heading (sideways / oblique / opposite / …).
* `dx, dy` — the candidate heading for this row (unit vector).
* `cos` — cosine between this heading and the true direction (graded goodness).
* `sheep_dx, sheep_dy, dist` — the true sheep offset (metadata for analysis).
* `t_0 …` / `e_0 …` — flattened **terrain** / **entity** channels (`arr.reshape(WIN, WIN)`).

Next: **`train_model_fox.ipynb`** trains a CNN+FNN direction *scorer* on these labels and
recovers the correct movement heading, confirming the model learns to follow the sheep.
Use **`fvision_play.py`** to flip through the rows and see each heading over the vision.